# Approaching (Almost) Any NLP Problem on Kaggle

In this post I'll talk about approaching natural language processing problems on Kaggle. As an example, we will use the data rom this competition. We will create a very basic first model first and then improve it using different other features. We will also see how deep neural networks can be used and end this post with some ideas about ensembling in general.

**This covers:**

* tfidf
* count features
* logistic regression
* naive bayes
* svm
* xgboost
* grid search
* word vectors
* LSTM
* GRU
* Ensembling

*NOTE*: This notebook is not meant for achieving a very high score on the Leaderboard for this dataset. However, if you follow it properly, you can get a very high score with some tuning. ;)

So, without wasting any time, let's start with importing some important python modules that I'll be using.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from tqdm import tqdm
from sklearn.svm import SVC
from keras.models import Sequential
from keras.layers import LSTM, GRU
from keras.layers import Dense, Activation, Dropout
from keras.layers import Embedding
from keras.layers import BatchNormalization
from keras.utils import to_categorical
from sklearn import preprocessing, decomposition, model_selection, metrics, pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from keras.layers import GlobalMaxPool1D, Conv1D, MaxPool1D, Flatten, Bidirectional, SpatialDropout1D
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.callbacks import EarlyStopping
from nltk import word_tokenize
from nltk.corpus import stopwords
stop_words = stopwords.words('english')

Let's load the datasets

In [2]:
train = pd.read_csv('./input/spooky/train.csv')
test = pd.read_csv('./input/spooky/test.csv')
sample = pd.read_csv('./input/spooky/sample_submission.csv')

A quick look at the data

In [3]:
train.head()

,id,text,author
0,id26305,"This process, however, afforded me no means of...",EAP
1,id17569,It never once occurred to me that the fumbling...,HPL
2,id11008,"In his left hand was a gold snuff box, from wh...",EAP
3,id27763,How lovely is spring As we looked from Windsor...,MWS
4,id12958,"Finding nothing else, not even gold, the Super...",HPL


In [4]:
test.head()

,id,text
0,id02310,"Still, as I urged our leaving Ireland with suc..."
1,id24541,"If a fire wanted fanning, it could readily be ..."
2,id00134,And when they had broken down the frail door t...
3,id27757,While I was thinking how I should possibly man...
4,id04081,I am not sure to what limit his knowledge may ...


In [5]:
sample.head()

,id,EAP,HPL,MWS
0,id02310,0.403494,0.287808,0.308698
1,id24541,0.403494,0.287808,0.308698
2,id00134,0.403494,0.287808,0.308698
3,id27757,0.403494,0.287808,0.308698
4,id04081,0.403494,0.287808,0.308698


The problem requires us to predict the author, i.e. EAP, HPL and MWS given the text. In simpler words, text classification with 3 different classes.

For this particular problem, Kaggle has specified multi-class log-loss as evaluation metric. This is implemented in the follow way (taken from: https://github.com/dnouri/nolearn/blob/master/nolearn/lasagne/util.py)

In [3]:
def multiclass_logloss(actual, predicted, eps=1e-15):
    '''Multi class version of Logarithmic Loss metric.
    :param actual: Array containing the actual target classes
    :param predicted: Matrix with class predictions, one probability per class
    '''
    if len(actual.shape) == 1:
        actual2 = np.zeros((actual.shape[0], predicted.shape[1]))
        for i, val in enumerate(actual):
            actual2[i, val] = 1
        actual = actual2

    clip = np.clip(predicted, eps, 1 - eps)
    rows = actual.shape[0]
    vsota = np.sum(actual * np.log(clip))
    return -1.0 / rows * vsota

We use the LabelEncoder from scikit-learn to convert text labels to integers, 0, 1, 2

In [4]:
lbl_enc = preprocessing.LabelEncoder()
y = lbl_enc.fit_transform(train.author.values)

Before going further it is important that we split the data into training and validation sets. We can do it using `train_test_split` from `model_selection` module of scikit-learn.

In [5]:
xtrain, xvalid, ytrain, yvalid = train_test_split(train.text.values, y, stratify=y, random_state=42, test_size=0.1, shuffle=True)

In [6]:
print(xtrain.shape)
print(xvalid.shape)

(17621,)
(1958,)


## Building Basic Models

Let's start building our very first model.

Our very first model si a simple TF-IDF (Term Frequency - Inverse Document Frequency) followed by a simple Logistic Regression.

In [26]:
tfv = TfidfVectorizer(min_df=3, max_features=None, strip_accents='unicode', analyzer='word', token_pattern=r'\w{1,}',
                      ngram_range=(1, 3), use_idf=True, smooth_idf=True, sublinear_tf=True, stop_words='english')

tfv.fit(list(xtrain) + list(xvalid))
xtrain_tfv = tfv.transform(xtrain)
xvalid_tfv = tfv.transform(xvalid)

In [11]:
clf = LogisticRegression(C=1.0)
clf.fit(xtrain_tfv, ytrain)
predictions = clf.predict_proba(xvalid_tfv)

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

logloss: 0.570


And there we go. We have our first model with a multiclass logloss of 0.570.

But we are greedy and want a better score. Lets look at the same model with a different data.

Instead of using TF-IDF, we can also use word counts as features. This can be done easily using CountVectorizer from scikit-learn.

In [27]:
ctv = CountVectorizer(analyzer='word', token_pattern=r'\w{1,}', ngram_range=(1, 3), stop_words = 'english')

ctv.fit(list(xtrain) + list(xvalid))
xtrain_ctv = ctv.transform(xtrain)
xvalid_ctv = ctv.transform(xvalid)

In [13]:
clf = LogisticRegression(C=1.0)
clf.fit(xtrain_ctv, ytrain)
predictions = clf.predict_proba(xvalid_ctv)

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

logloss: 0.527


Aaaaanddddddd Wallah! We just improved our model by 0.04!!!

Next, let's try a very simple model which was quite famous in ancient times - Naive Bayes.

Let's see what happens when we use naive bayes on these two datasets:

In [14]:
clf = MultinomialNB()
clf.fit(xtrain_tfv, ytrain)
predictions = clf.predict_proba(xvalid_tfv)

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

logloss: 0.578


Good performance! But the logistic regression on counts is still better! What happens when we use this model on counts data instead?

In [15]:
clf = MultinomialNB()
clf.fit(xtrain_ctv, ytrain)
predictions = clf.predict_proba(xvalid_ctv)

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

logloss: 0.485


Whoa! Seems like old stuff still works good!!!! One more ancient algorithms in the list is SVMs. Some people "love" SVMs. So, we must try SVM on this dataset.

Since SVMs take a lot of time, we will reduce the number of features from the TF-IDF using Singular Value Decomposition before applying SVM.

Also, note that before applying SVMs, we *must* standardize the data.

In [16]:
svd = decomposition.TruncatedSVD(n_components=120)
svd.fit(xtrain_tfv)
xtrain_svd = svd.transform(xtrain_tfv)
xvalid_svd = svd.transform(xvalid_tfv)

scl = preprocessing.StandardScaler()
scl.fit(xtrain_svd)
xtrain_svd_scl = scl.transform(xtrain_svd)
xvalid_svd_scl = scl.transform(xvalid_svd)

Now it's time to apply SVM. After running the following cell, feel free to go for a walk or talk to your girlfriend/boyfriend. :P

In [17]:
clf = SVC(C=1.0, probability=True)
clf.fit(xtrain_svd_scl, ytrain)
predictions = clf.predict_proba(xvalid_svd_scl)

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

logloss: 0.728


Oops! time to get up! Looks like SVM doesn't perform well on this data...!

Before moving further, lets apply the most popular algorithm on Kaggle: xgboost!

In [18]:
clf = xgb.XGBClassifier(max_depth=7, n_estimators=200, colsample_bytree=0.8, subsample=0.8, nthread=10, learning_rate=0.1)
clf.fit(xtrain_tfv.tocsc(), ytrain)
predictions = clf.predict_proba(xvalid_tfv.tocsc())

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

logloss: 0.780


In [19]:
clf = xgb.XGBClassifier(max_depth=7, n_estimators=200, colsample_bytree=0.8, subsample=0.8, nthread=10, learning_rate=0.1)
clf.fit(xtrain_ctv.tocsc(), ytrain)
predictions = clf.predict_proba(xvalid_ctv.tocsc())

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

logloss: 0.773


In [20]:
clf = xgb.XGBClassifier(max_depth=7, n_estimators=200, colsample_bytree=0.8, subsample=0.8, nthread=10, learning_rate=0.1)
clf.fit(xtrain_svd, ytrain)
predictions = clf.predict_proba(xvalid_svd)

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

logloss: 0.768


In [21]:
clf = xgb.XGBClassifier(nthread=10)
clf.fit(xtrain_svd, ytrain)
predictions = clf.predict_proba(xvalid_svd)

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

logloss: 0.773


Seems like no luck with XGBoost! But that is not correct. I haven't done any hyperparameter optimizations yet. And since I'm lazy, I'll just tell you how to do it and you can do it on your own! ;). This will be discussed in the next section:

## Grid Search

It's a technique for hyperparameter optimization. Not so effective but can give good results if you know the grid you want to use. I specify the parameters that should usually be used in this post: http://blog.kaggle.com/2016/07/21/approaching-almost-any-machine-learning-problem-abhishek-thakur/ Please keep in mind that these are the parameters I usually use. There are many other methods of hyperparameter optimization which may or may not be as effective.

In this section, I'll talk about grid search using logistic regression.

Before starting with grid search we need to create a scoring function. This is accomplished using the `make_scorer` function of scikit-learn.

In [22]:
mll_scorer = metrics.make_scorer(multiclass_logloss, greater_is_better=False, response_method='predict_proba')

Next we need a pipeline. For demonstration here, I'll be using a pipeline consisting of SVD, scaling and then logistic regression. It's better to understand with more modules in pipeline than just one ;)

In [23]:
svd = TruncatedSVD()

scl = preprocessing.StandardScaler()

lr_model = LogisticRegression(solver='liblinear')

clf = pipeline.Pipeline([('svd', svd),
                         ('scl', scl),
                         ('lr', lr_model)])

Next we need a grid of parameters:

In [24]:
param_grid = {'svd__n_components': [120, 180],
              'lr__C': [0.1, 1.0, 10],
              'lr__penalty': ['l1', 'l2']}

So, for SVD we evaluate 120 and 180 components and for logistic regression we evaluate three different values of C with l1 and l2 penalty. We can now start grid search on these parameters.

In [25]:
model = GridSearchCV(estimator=clf, param_grid=param_grid, scoring=mll_scorer, verbose=10, n_jobs=-1, refit=True, cv=2)

model.fit(xtrain_tfv, ytrain)
print('Best score: %0.3f' % model.best_score_)
print('Best parameters set:')
best_parameters = model.best_estimator_.get_params()
for param_name in sorted(param_grid.keys()):
    print('\t%s: %r' % (param_name, best_parameters[param_name]))

Fitting 2 folds for each of 12 candidates, totalling 24 fits
Best score: -0.743
Best parameters set:
	lr__C: 1.0
	lr__penalty: 'l2'
	svd__n_components: 180


The score comes similar to what we had for SVM. This technique can be used to finetune xgboost or evne multinomial naive bayes as below. We will use the tfidf data here:

In [26]:
nb_model = MultinomialNB()

clf = pipeline.Pipeline([('nb', nb_model)])

param_grid = {'nb__alpha': [0.001, 0.01, 0.1, 1, 10, 100]}

model = GridSearchCV(estimator=clf, param_grid=param_grid, scoring=mll_scorer, verbose=10, n_jobs=-1, refit=True, cv=2)

model.fit(xtrain_tfv, ytrain)
print('Best score: %0.3f' % model.best_score_)
print('Best parameters set:')
best_parameters = model.best_estimator_.get_params()
for param_name in sorted(param_grid.keys()):
    print('\t%s: %r' % (param_name, best_parameters[param_name]))

Fitting 2 folds for each of 6 candidates, totalling 12 fits
Best score: -0.492
Best parameters set:
	nb__alpha: 0.1


This is an improvement of 8% over the original naive bayes score!

In NLP problems, it's customary to look at word vectors. Word vectors give a lot of insights about the data. Let's dive into that.

## Word Vectors

Without going into too much details, I would explain how to create sentence vectors and how can we use them to create a machine learning model on top of it. I am a fan of GloVe vectors, word2vec and fasttext. In this post, I'll be using the GloVe vectors. You can download the GloVe vectors from here `http://www-nlp.stanford.edu/data/glove/840B.300d.zip`

In [7]:
embeddings_index = {}
f = open('./input/glove.840B.300d.txt')
for line in tqdm(f):
    values = line.split(' ')
    word = values[0]
    coefs = np.asarray(values[1:], dtype='float32')
    embeddings_index[word] = coefs
f.close()

print('Found %s word vectors.' % len(embeddings_index))

2196017it [01:53, 19363.36it/s]

Found 2196016 word vectors.


In [8]:
def sent2vec(s):
    words = str(s).lower()
    words = word_tokenize(words)
    words = [w for w in words if not w in stop_words]
    words = [w for w in words if w.isalpha()]
    M = []
    for w in words:
        try:
            M.append(embeddings_index[w])
        except:
            continue
    M = np.array(M)
    v = M.sum(axis=0)
    if type(v) != np.ndarray:
        return np.zeros(300)
    return v / np.sqrt((v ** 2).sum())
    

In [9]:
xtrain_glove = [sent2vec(x) for x in tqdm(xtrain)]
xvalid_glove = [sent2vec(x) for x in tqdm(xvalid)]

100%|██████████| 1958/1958 [00:00<00:00, 6143.10it/s]


In [10]:
xtrain_glove = np.array(xtrain_glove)
xvalid_glove = np.array(xvalid_glove)

Let's see the performance of xgboost on glove features:

In [11]:
clf = xgb.XGBClassifier(nthread=10, silent=False)
clf.fit(xtrain_glove, ytrain)
predictions = clf.predict_proba(xvalid_glove)

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [10:48:15] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


KeyboardInterrupt: 

In [ ]:
clf = xgb.XGBClassifier(max_depth=7, n_estimators=200, colsample_bytree=0.8,
                        subsample=0.8, nthread=10, learning_rate=0.1, silent=False)
clf.fit(xtrain_glove, ytrain)
predictions = clf.predict_proba(xvalid_glove)

print('logloss: %0.3f' % multiclass_logloss(yvalid, predictions))

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [09:45:59] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


logloss: 0.686


We see that a simple tuning of parameters can improve xgboost score on GloVe features! Believe me you can squeeze a lot more from it.

## Deep Learning

But this is an era of deep learning! We can't live without training a few neural networks. Here, we will train LSTM and a simple dense network on the GloVe features. Let's start with the dense network first:

In [12]:
scl = preprocessing.StandardScaler()
xtrain_glove_scl = scl.fit_transform(xtrain_glove)
xvalid_glove_scl = scl.transform(xvalid_glove)

In [13]:
ytrain_enc = to_categorical(ytrain)
yvalid_enc = to_categorical(yvalid)

In [14]:
model = Sequential()

model.add(Dense(300, input_dim=300, activation='relu'))
model.add(Dropout(0.2))
model.add(BatchNormalization())

model.add(Dense(300, activation='relu'))
model.add(Dropout(0.3))
model.add(BatchNormalization())

model.add(Dense(3))
model.add(Activation('softmax'))

model.compile(loss='categorical_crossentropy', optimizer='adam')

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [15]:
model.fit(xtrain_glove_scl, y=ytrain_enc, batch_size=64, epochs=5, verbose=1,
          validation_data=(xvalid_glove_scl, yvalid_enc))

Epoch 1/5
276/276 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.9065 - val_loss: 0.7333
Epoch 2/5
276/276 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 0.7009 - val_loss: 0.6885
Epoch 3/5
276/276 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.6445 - val_loss: 0.6726
Epoch 4/5
276/276 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.5977 - val_loss: 0.6574
Epoch 5/5
276/276 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 0.5643 - val_loss: 0.6572


You need to keep on tuning the parameters of the neural network, add more layers, increase dropout to get better results. Here, I'm just showing that its fast to implement and run and gets better result than xgboost without any optimization :)

To move further, i.e. with LSTMs we need to tokenize the text data

In [16]:
token = Tokenizer(num_words=None)
max_len = 70

token.fit_on_texts(list(xtrain) + list(xvalid))
xtrain_seq = token.texts_to_sequences(xtrain)
xvalid_seq = token.texts_to_sequences(xvalid)

xtrain_pad = pad_sequences(xtrain_seq, maxlen=max_len)
xvalid_pad = pad_sequences(xvalid_seq, maxlen=max_len)

word_index = token.word_index

In [17]:
embedding_matrix = np.zeros((len(word_index) + 1, 300))
for word, i in tqdm(word_index.items()):
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector

100%|██████████| 25943/25943 [00:00<00:00, 414871.18it/s]


In [18]:
model = Sequential()
model.add(Embedding(len(word_index) + 1,
                    300,
                    weights=[embedding_matrix],
                    input_length=max_len,
                    trainable=False))
model.add(SpatialDropout1D(0.3))
model.add(LSTM(100, dropout=0.3, recurrent_dropout=0.3))

model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.8))

model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.8))

model.add(Dense(3))
model.add(Activation('softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam')

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [19]:
model.fit(xtrain_pad, y=ytrain_enc, batch_size=512, epochs=100, verbose=2, validation_data=(xvalid_pad, yvalid_enc))

Epoch 1/100
35/35 - 9s - 257ms/step - loss: 1.0754 - val_loss: 0.9785
Epoch 2/100
35/35 - 5s - 150ms/step - loss: 0.9517 - val_loss: 0.7980
Epoch 3/100
35/35 - 5s - 149ms/step - loss: 0.8718 - val_loss: 0.7510
Epoch 4/100
35/35 - 5s - 148ms/step - loss: 0.8370 - val_loss: 0.7241
Epoch 5/100
35/35 - 5s - 150ms/step - loss: 0.8177 - val_loss: 0.7098
Epoch 6/100
35/35 - 5s - 149ms/step - loss: 0.7894 - val_loss: 0.7081
Epoch 7/100
35/35 - 5s - 149ms/step - loss: 0.7779 - val_loss: 0.6801
Epoch 8/100
35/35 - 5s - 149ms/step - loss: 0.7564 - val_loss: 0.6615
Epoch 9/100
35/35 - 5s - 149ms/step - loss: 0.7450 - val_loss: 0.6503
Epoch 10/100
35/35 - 5s - 150ms/step - loss: 0.7266 - val_loss: 0.6399
Epoch 11/100
35/35 - 5s - 150ms/step - loss: 0.7073 - val_loss: 0.6354
Epoch 12/100
35/35 - 5s - 148ms/step - loss: 0.6960 - val_loss: 0.6118
Epoch 13/100
35/35 - 5s - 149ms/step - loss: 0.6752 - val_loss: 0.6130
Epoch 14/100
35/35 - 5s - 153ms/step - loss: 0.6548 - val_loss: 0.5975
Epoch 15/100
35

We see that the score is not less than 0.5. I ran it for many epochs without stopping at the best but you can use early stopping to stop at the best iteration. How do I use early stopping?

Well, pretty easy. Let's compile the model again:

In [20]:
model = Sequential()
model.add(Embedding(len(word_index) + 1,
                    300,
                    weights=[embedding_matrix],
                    input_length=max_len,
                    trainable=False))
model.add(SpatialDropout1D(0.3))
model.add(LSTM(300, dropout=0.3, recurrent_dropout=0.3))

model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.8))

model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.8))

model.add(Dense(3))
model.add(Activation('softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam')

earlystop = EarlyStopping(monitor='val_loss', min_delta=0, patience=3, verbose=0, mode='auto')
model.fit(xtrain_pad, y=ytrain_enc, batch_size=512, epochs=100, verbose=2, validation_data=(xvalid_pad, yvalid_enc), callbacks=[earlystop])

Epoch 1/100


c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


35/35 - 80s - 2s/step - loss: 1.0674 - val_loss: 0.9028
Epoch 2/100
35/35 - 98s - 3s/step - loss: 0.9133 - val_loss: 0.7818
Epoch 3/100
35/35 - 14s - 404ms/step - loss: 0.8589 - val_loss: 0.7575
Epoch 4/100
35/35 - 45s - 1s/step - loss: 0.8318 - val_loss: 0.7523
Epoch 5/100
35/35 - 17s - 489ms/step - loss: 0.8079 - val_loss: 0.7291
Epoch 6/100
35/35 - 15s - 421ms/step - loss: 0.7882 - val_loss: 0.6997
Epoch 7/100
35/35 - 15s - 418ms/step - loss: 0.7711 - val_loss: 0.6886
Epoch 8/100
35/35 - 15s - 415ms/step - loss: 0.7493 - val_loss: 0.6599
Epoch 9/100
35/35 - 15s - 415ms/step - loss: 0.7271 - val_loss: 0.6593
Epoch 10/100
35/35 - 14s - 388ms/step - loss: 0.7158 - val_loss: 0.6269
Epoch 11/100
35/35 - 14s - 408ms/step - loss: 0.6861 - val_loss: 0.6408
Epoch 12/100
35/35 - 15s - 418ms/step - loss: 0.6697 - val_loss: 0.6078
Epoch 13/100
35/35 - 15s - 418ms/step - loss: 0.6589 - val_loss: 0.6190
Epoch 14/100
35/35 - 15s - 418ms/step - loss: 0.6417 - val_loss: 0.5903
Epoch 15/100
35/35 - 1

One question could be: why do I use so much dropout? Well, fit the model with no or little dropout and you will that it starts to overfit :)

Let's see if Bi-directional LSTM can give us better results. It's a piece of cake to do it with Keras :)

In [21]:
model = Sequential()
model.add(Embedding(len(word_index) + 1,
                    300,
                    weights=[embedding_matrix],
                    input_length=max_len,
                    trainable=False))
model.add(SpatialDropout1D(0.3))
model.add(Bidirectional(LSTM(300, dropout=0.3, recurrent_dropout=0.3)))

model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.8))

model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.8))

model.add(Dense(3))
model.add(Activation('softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam')

earlystop = EarlyStopping(monitor='val_loss', min_delta=0, patience=3, verbose=0, mode='auto')
model.fit(xtrain_pad, y=ytrain_enc, batch_size=512, epochs=100, verbose=2, validation_data=(xvalid_pad, yvalid_enc), callbacks=[earlystop])

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/100
35/35 - 30s - 861ms/step - loss: 1.0621 - val_loss: 0.8928
Epoch 2/100
35/35 - 25s - 702ms/step - loss: 0.9031 - val_loss: 0.7900
Epoch 3/100
35/35 - 24s - 695ms/step - loss: 0.8524 - val_loss: 0.7527
Epoch 4/100
35/35 - 24s - 696ms/step - loss: 0.8191 - val_loss: 0.7168
Epoch 5/100
35/35 - 24s - 698ms/step - loss: 0.7949 - val_loss: 0.7098
Epoch 6/100
35/35 - 24s - 696ms/step - loss: 0.7737 - val_loss: 0.6893
Epoch 7/100
35/35 - 24s - 696ms/step - loss: 0.7586 - val_loss: 0.6897
Epoch 8/100
35/35 - 24s - 697ms/step - loss: 0.7403 - val_loss: 0.6660
Epoch 9/100
35/35 - 24s - 697ms/step - loss: 0.7214 - val_loss: 0.6444
Epoch 10/100
35/35 - 24s - 694ms/step - loss: 0.7022 - val_loss: 0.6318
Epoch 11/100
35/35 - 24s - 696ms/step - loss: 0.6867 - val_loss: 0.6170
Epoch 12/100
35/35 - 24s - 695ms/step - loss: 0.6574 - val_loss: 0.5994
Epoch 13/100
35/35 - 24s - 695ms/step - loss: 0.6427 - val_loss: 0.5990
Epoch 14/100
35/35 - 24s - 698ms/step - loss: 0.6204 - val_loss: 0.5649
E

Pretty close! Let's try two layers of GRU:

In [22]:
model = Sequential()
model.add(Embedding(len(word_index) + 1,
                    300,
                    weights=[embedding_matrix],
                    input_length=max_len,
                    trainable=False))
model.add(SpatialDropout1D(0.3))
model.add(GRU(300, dropout=0.3, recurrent_dropout=0.3, return_sequences=True))
model.add(GRU(300, dropout=0.3))

model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.8))

model.add(Dense(1024, activation='relu'))
model.add(Dropout(0.8))

model.add(Dense(3))
model.add(Activation('softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam')

earlystop = EarlyStopping(monitor='val_loss', min_delta=0, patience=3, verbose=0, mode='auto')
model.fit(xtrain_pad, y=ytrain_enc, batch_size=512, epochs=100, verbose=2, validation_data=(xvalid_pad, yvalid_enc), callbacks=[earlystop])

Epoch 1/100


c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


35/35 - 30s - 851ms/step - loss: 1.0820 - val_loss: 0.9524
Epoch 2/100
35/35 - 24s - 685ms/step - loss: 0.9433 - val_loss: 0.8159
Epoch 3/100
35/35 - 24s - 695ms/step - loss: 0.8760 - val_loss: 0.8044
Epoch 4/100
35/35 - 24s - 697ms/step - loss: 0.8405 - val_loss: 0.7583
Epoch 5/100
35/35 - 26s - 747ms/step - loss: 0.8079 - val_loss: 0.7169
Epoch 6/100
35/35 - 26s - 754ms/step - loss: 0.7836 - val_loss: 0.7440
Epoch 7/100
35/35 - 26s - 749ms/step - loss: 0.7590 - val_loss: 0.6926
Epoch 8/100
35/35 - 26s - 753ms/step - loss: 0.7365 - val_loss: 0.6597
Epoch 9/100
35/35 - 26s - 739ms/step - loss: 0.7125 - val_loss: 0.6590
Epoch 10/100
35/35 - 27s - 759ms/step - loss: 0.6971 - val_loss: 0.6421
Epoch 11/100
35/35 - 27s - 760ms/step - loss: 0.6757 - val_loss: 0.6169
Epoch 12/100
35/35 - 27s - 768ms/step - loss: 0.6631 - val_loss: 0.6154
Epoch 13/100
35/35 - 27s - 776ms/step - loss: 0.6427 - val_loss: 0.5745
Epoch 14/100
35/35 - 27s - 776ms/step - loss: 0.6181 - val_loss: 0.5605
Epoch 15/100


Nice! Much better than what we had previously! Keep optimizing and the performance will keep improving. Worth trying: stemming and lemmatization. This is something I'm skipping for now.

In the Kaggle world, to get a top score you should have an ensemble of models. Let's check a little bit of ensembling!

## Ensembling

Few months back I made a simple ensembler but I didn't have time to develop it fully. It can be found here: https://github.com/abhishekkrthakur/pysembler. I'm going to use some part of it here:

In [24]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, KFold
import pandas as pd
import os
import sys
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="[%9asctime)s] %(levelname)s %(message)s",
    datefmt='%H:%M:%S', stream=sys.stdout
)
logger = logging.getLogger(__name__)

class Ensembler(object):
    def __init__(self, model_dict, num_folds=3, task_type='classification', optimize=roc_auc_score, lower_is_better=False, save_path=None):
        '''
        Ensembler init function
        :param model_dict: model dictionary, see README for its format
        :param num_folds: the number of folds for ensembling
        :param task_type: classification or regression
        :param optimize: the function to optimize for, e.g. AUC, logloss, etc. Must have two arguments y_test and y_pred
        :param lower_is_better: is lower value of optimization function better or higher
        :param save_path: path to which model pickles will be dumped to along with generated predictions, or None
        '''
        self.model_dict = model_dict
        self.levels = len(self.model_dict)
        self.num_folds = num_folds
        self.task_type = task_type
        self.optimize = optimize
        self.lower_is_better = lower_is_better
        self.save_path = save_path

        self.training_data = None
        self.test_data = None
        self.y = None
        self.lbl_enc = None
        self.y_enc = None
        self.train_prediction_dict = None
        self.test_prediction_dict = None
        self.num_classes = None

    def fit(self, training_data, y, lentrain):
        '''
        :param training_data: training data in tabular format
        :param y: binary, multi-class or regression
        :return: chain of models to be used in prediction
        '''

        self.training_data = training_data
        self.y = y

        if self.task_type == 'classification':
            self.num_classes = len(np.unique(self.y))
            logger.info('Found %d classes', self.num_classes)
            self.lbl_enc = LabelEncoder()
            self.y_enc = self.lbl_enc.fit_transform(self.y)
            kf = StratifiedKFold(n_splits=self.num_folds)
            train_prediction_shape = (lentrain, self.num_classes)
        else:
            self.num_classes = -1
            self.y_enc = self.y
            kf = KFold(n_splits=self.num_folds)
            train_prediction_shape = (lentrain, 1)

        self.train_prediction_dict = {}
        for level in range(self.levels):
            self.train_prediction_dict[level] = np.zeros((train_prediction_shape[0], train_prediction_shape[1] * len(self.model_dict[level])))

        for level in range(self.levels):
            if level == 0:
                temp_train = self.training_data
            else:
                temp_train = self.train_prediction_dict[level - 1]

            for model_num, model in enumerate(self.model_dict[level]):
                validation_scores = []
                foldnum = 1
                for train_index, valid_index in kf.split(self.train_prediction_dict[0], self.y_enc):
                    logger.info('Training Level %d Fold # %d. Model # %d', level, foldnum, model_num)

                    if level != 0:
                        l_training_data = temp_train[train_index]
                        l_validation_data = temp_train[valid_index]
                        model.fit(l_training_data, self.y_enc[train_index])
                    else:
                        l0_training_data = temp_train[0][model_num]
                        if type(l0_training_data) == list:
                            l_training_data = [x[train_index] for x in l0_training_data]
                            l_validation_data = [x[valid_index] for x in l0_training_data]
                        else:
                            l_training_data = l0_training_data[train_index]
                            l_validation_data = l0_training_data[valid_index]
                        model.fit(l_training_data, self.y_enc[train_index])

                    logger.info('Predicting Level %d. Fold # %d. Model # %d', level, foldnum, model_num)

                    if self.task_type == 'classification':
                        temp_train_predictions = model.predict_proba(l_validation_data)
                        self.train_prediction_dict[level][valid_index, (model_num * self.num_classes):(model_num * self.num_classes) + self.num_classes] = temp_train_predictions
                    else:
                        temp_train_predictions = model.predict(l_validation_data)
                        self.train_prediction_dict[level][valid_index, model_num] = temp_train_predictions
                    validation_score = self.optimize(self.y_enc[valid_index], temp_train_predictions)
                    validation_scores.append(validation_score)
                    logger.info('Level %d. Fold # %d. Model # %d. Validation Score = %f', level, foldnum, model_num, validation_score)
                    foldnum += 1
                avg_score = np.mean(validation_scores)
                std_score = np.std(validation_score)
                logger.info('Level %d. Model # %d. Mean Score = %f. Std Dev = %f', level, model_num, avg_score, std_score)

            logger.info('Saving predictions for level # %d', level)
            train_predictions_df = pd.DataFrame(self.train_prediction_dict[level])
            train_predictions_df.to_csv(os.path.join(self.save_path, 'train_predictions_level_' + str(level) + '.csv'), index=False, header=None)

        return self.train_prediction_dict

    def predict(self, test_data, lentest):
        self.test_data = test_data
        if self.task_type == 'classification':
            test_prediction_shape = (lentest, self.num_classes)
        else:
            test_prediction_shape = (lentest, 1)

        self.test_prediction_dict = {}
        for level in range(self.levels):
            self.test_prediction_dict[level] = np.zeros((test_prediction_shape[0], test_prediction_shape[1] * len(self.model_dict[level])))
        self.test_data = test_data
        for level in range(self.levels):
            if level == 0:
                temp_train = self.training_data
                temp_test = self.test_data
            else:
                temp_train = self.train_prediction_dict[level - 1]
                temp_test = self.test_prediction_dict[level - 1]

            for model_num, model in enumerate(self.model_dict[level]):
                logger.info('Training Fulldata Level %d. Model # %d', level, model_num)
                if level == 0:
                    model.fit(temp_train[0][model_num], self.y_enc)
                else:
                    model.fit(temp_train, self.y_enc)

                logger.info('Predicting Test Level %d. Model # %d', level, model_num)

                if self.task_type == 'classification':
                    if level == 0:
                        temp_test_predictions = model.predict_proba(temp_test[0][model_num])
                    else:
                        temp_test_predictions = model.predict_proba(temp_test)
                    self.test_prediction_dict[level][:, (model_num * self.num_classes): (model_num * self.num_classes) + self.num_classes] = temp_test_predictions

                else:
                    if level == 0:
                        temp_test_predictions = model.predict(temp_test[0][model_num])
                    else:
                        temp_test_predictions = model.predict(temp_test)
                    self.test_prediction_dict[level][:, model_num] = temp_test_predictions

            test_predictions_df = pd.DataFrame(self.test_prediction_dict[level])
            test_predictions_df.to_csv(os.path.join(self.save_path, 'test_predictions_level_' + str(level) + '.csv'), index=False, header=None)

        return self.test_prediction_dict

In [28]:
train_data_dict = {0: [xtrain_tfv, xtrain_ctv, xtrain_tfv, xtrain_ctv], 1: [xtrain_glove]}
test_data_dict = {0: [xvalid_tfv, xvalid_ctv, xvalid_tfv, xvalid_ctv], 1: [xvalid_glove]}

model_dict = {0: [LogisticRegression(), LogisticRegression(), MultinomialNB(alpha=0.1), MultinomialNB()],
              1: [xgb.XGBClassifier(silent=True, n_estimators=120, max_depth=7)]}

ens = Ensembler(model_dict=model_dict, num_folds=3, task_type='classification',
                optimize=multiclass_logloss, lower_is_better=True, save_path='')

ens.fit(train_data_dict, ytrain, lentrain=xtrain_glove.shape[0])
preds = ens.predict(test_data_dict, lentest=xvalid_glove.shape[0])

[{'name': '__main__', 'msg': 'Found %d classes', 'args': (3,), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'C:\\Users\\USER\\AppData\\Local\\Temp\\ipykernel_16768\\1736488997.py', 'filename': '1736488997.py', 'module': '1736488997', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 57, 'funcName': 'fit', 'created': 1787885267.1308222, 'msecs': 130.0, 'relativeCreated': 3742984.8251342773, 'thread': 17296, 'threadName': 'MainThread', 'processName': 'MainProcess', 'process': 16768, 'message': 'Found 3 classes'}sctime)s] INFO Found 3 classes
[{'name': '__main__', 'msg': 'Training Level %d Fold # %d. Model # %d', 'args': (0, 1, 0), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'C:\\Users\\USER\\AppData\\Local\\Temp\\ipykernel_16768\\1736488997.py', 'filename': '1736488997.py', 'module': '1736488997', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 82, 'funcName': 'fit', 'created': 1787885267.1358223, 'msecs': 135.0, 'relativeCreated': 3742989.8252487

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:48:13] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[{'name': '__main__', 'msg': 'Predicting Level %d. Fold # %d. Model # %d', 'args': (1, 1, 0), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'C:\\Users\\USER\\AppData\\Local\\Temp\\ipykernel_16768\\1736488997.py', 'filename': '1736488997.py', 'module': '1736488997', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 98, 'funcName': 'fit', 'created': 1787885293.7986338, 'msecs': 798.0, 'relativeCreated': 3769652.6367664337, 'thread': 17296, 'threadName': 'MainThread', 'processName': 'MainProcess', 'process': 16768, 'message': 'Predicting Level 1. Fold # 1. Model # 0'}sctime)s] INFO Predicting Level 1. Fold # 1. Model # 0
[{'name': '__main__', 'msg': 'Level %d. Fold # %d. Model # %d. Validation Score = %f', 'args': (1, 1, 0, np.float64(0.49383039650063376)), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'C:\\Users\\USER\\AppData\\Local\\Temp\\ipykernel_16768\\1736488997.py', 'filename': '1736488997.py', 'module': '1736488997', 'exc_info': None, 'exc_text': None, 'stack_

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:48:14] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[{'name': '__main__', 'msg': 'Predicting Level %d. Fold # %d. Model # %d', 'args': (1, 3, 0), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'C:\\Users\\USER\\AppData\\Local\\Temp\\ipykernel_16768\\1736488997.py', 'filename': '1736488997.py', 'module': '1736488997', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 98, 'funcName': 'fit', 'created': 1787885295.0618167, 'msecs': 61.0, 'relativeCreated': 3770915.819644928, 'thread': 17296, 'threadName': 'MainThread', 'processName': 'MainProcess', 'process': 16768, 'message': 'Predicting Level 1. Fold # 3. Model # 0'}sctime)s] INFO Predicting Level 1. Fold # 3. Model # 0
[{'name': '__main__', 'msg': 'Level %d. Fold # %d. Model # %d. Validation Score = %f', 'args': (1, 3, 0, np.float64(0.5024428623975651)), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'C:\\Users\\USER\\AppData\\Local\\Temp\\ipykernel_16768\\1736488997.py', 'filename': '1736488997.py', 'module': '1736488997', 'exc_info': None, 'exc_text': None, 'stack_inf

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:200: UserWarning: [11:48:24] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "silent" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[{'name': '__main__', 'msg': 'Predicting Test Level %d. Model # %d', 'args': (1, 0), 'levelname': 'INFO', 'levelno': 20, 'pathname': 'C:\\Users\\USER\\AppData\\Local\\Temp\\ipykernel_16768\\1736488997.py', 'filename': '1736488997.py', 'module': '1736488997', 'exc_info': None, 'exc_text': None, 'stack_info': None, 'lineno': 146, 'funcName': 'predict', 'created': 1787885304.7454484, 'msecs': 745.0, 'relativeCreated': 3780599.451303482, 'thread': 17296, 'threadName': 'MainThread', 'processName': 'MainProcess', 'process': 16768, 'message': 'Predicting Test Level 1. Model # 0'}sctime)s] INFO Predicting Test Level 1. Model # 0


In [29]:
multiclass_logloss(yvalid, preds[1])

np.float64(0.481769085260825)

Thus, we see that ensembling improves the score by a great extent! Since this is supposed to be a tutorial only I won't be providing any CSVs that you can submit to the leaderboard.

I hope you like it!

P.S.: If the response is good, I'll add more stuff in this! :)